In [ ]:
### balance the dataset to have equal number of exacerbated and non-exacerbated samples if needed

In [ ]:
import pandas as pd

df8918_exacerbated_0_1 = pd.read_pickle('********.pkl')
df8918_exacerbated_0_1

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
from torch.nn.utils import clip_grad_norm_
import wandb  # 新增
import os
import pandas as pd

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

df_copd_asthma_speech = df8918_exacerbated_0_1.copy()


features = df_copd_asthma_speech.iloc[:, :54].values.astype(np.float32)
emotion_labels = df_copd_asthma_speech['event_clinical_yesno'].values.astype(np.int64)
speaker_ids = df_copd_asthma_speech['sid'].values

In [ ]:
le_speaker = LabelEncoder()
speaker_labels = le_speaker.fit_transform(speaker_ids).astype(np.int64)
num_speakers = len(le_speaker.classes_)


unique_speakers = np.unique(speaker_labels)
speaker_to_idx = {spk: [] for spk in unique_speakers}
for i, spk in enumerate(speaker_labels):
    speaker_to_idx[spk].append(i)

#Step 1: Set aside a test set 
train_val_speakers, test_speakers = train_test_split(unique_speakers, test_size=0.2, random_state=42)
#Step 2: Divide the remaining 80% into a training set and a validation set 
train_speakers, val_speakers = train_test_split(train_val_speakers, test_size=0.2, random_state=42)

# Get indexes for training, validation, and test sets based on speaker IDs
train_indices = [i for spk in train_speakers for i in speaker_to_idx[spk]]
val_indices = [i for spk in val_speakers for i in speaker_to_idx[spk]]
test_indices = [i for spk in test_speakers for i in speaker_to_idx[spk]]

X_train, y_emo_train, y_spk_train = features[train_indices], emotion_labels[train_indices], speaker_labels[train_indices]
X_val, y_emo_val, y_spk_val = features[val_indices], emotion_labels[val_indices], speaker_labels [val_indices]
X_test, y_emo_test, y_spk_test = features[test_indices], emotion_labels[test_indices], speaker_labels[test_indices]


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
class MultiTaskDataset(Dataset):
    def __init__(self, features, emotion_labels, speaker_labels):
        self.features = torch.tensor(features)
        self.emotion_labels = torch.tensor(emotion_labels)
        self.speaker_labels = torch.tensor(speaker_labels)
    def __len__(self): return len(self.emotion_labels)
    def __getitem__(self, idx): return self.features[idx], self.emotion_labels[idx], self.speaker_labels[idx]

train_loader = DataLoader(MultiTaskDataset(X_train, y_emo_train, y_spk_train), batch_size=32, shuffle=True)
val_loader = DataLoader(MultiTaskDataset(X_val, y_emo_val, y_spk_val), batch_size=32, shuffle=False)
test_loader = DataLoader(MultiTaskDataset(X_test, y_emo_test, y_spk_test), batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# count the number of samples for each class in the training set
class_counts = np.bincount(y_emo_test)
class_counts

In [ ]:
# GRL
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_param):
        ctx.lambda_param = lambda_param
        return x.clone()
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_param * grad_output, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_param=1.0):
        super().__init__()
        self.lambda_param = lambda_param
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_param)

In [ ]:
def compute_j_ratio(embeddings, labels):
    classes = np.unique(labels)
    if len(classes) < 2: return 0.0
    D = embeddings.shape[1]
    overall_mean = np.mean(embeddings, axis=0)
    S_B, S_W = np.zeros((D, D)), np.zeros((D, D))
    for c in classes:
        X_c = embeddings[labels == c]
        mean_c = np.mean(X_c, axis=0)
        n_c = X_c.shape[0]
        diff = (mean_c - overall_mean).reshape(-1, 1)
        S_B += n_c * diff @ diff.T
        cov = np.cov(X_c, rowvar=False)
        S_W += (n_c - 1) * cov
    return np.trace(S_B) / (np.trace(S_W) + 1e-8)

def extract_embeddings(model, loader, device):
    model.eval()
    embs, spks = [], []
    with torch.no_grad():
        for feats, _, spk in loader:
            feats = feats.to(device)
            emb = model(feats)
            embs.append(emb.cpu().numpy())
            spks.append(spk.numpy())
    return np.concatenate(embs, 0), np.concatenate(spks, 0)

In [ ]:
# Model Definition (Shared)
class LocallyEnhancedFFN(nn.Module):
   def __init__(self, input_dim):
       super().__init__()
       self.conv1 = nn.Conv1d(input_dim, 64, kernel_size=1)
       self.depthwise = nn.Conv1d(64, 64, kernel_size=1, groups=64)
       self.conv2 = nn.Conv1d(64, input_dim, kernel_size=1)
       self.dropout = nn.Dropout(0.5)
   def forward(self, x):
       residual = x
       x = x.permute(0, 2, 1)
       x = self.conv1(x)
       x = self.depthwise(x)
       x = F.gelu(x)
       x = self.dropout(x)
       x = self.conv2(x)
       x = F.gelu(x)
       x = x.permute(0, 2, 1)
       return x + residual

class TransformerEncoderBlock(nn.Module):
   def __init__(self, embed_dim, num_heads):
       super().__init__()
       self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=0.5)
       self.norm1 = nn.LayerNorm(embed_dim)
       self.norm2 = nn.LayerNorm(embed_dim)
       self.ffn = LocallyEnhancedFFN(embed_dim)
       self.dropout = nn.Dropout(0.5)
   def forward(self, x):
       residual = x
       x_attn = x.permute(1, 0, 2)
       attn_output, _ = self.self_attn(x_attn, x_attn, x_attn)
       attn_output = attn_output.permute(1, 0, 2)
       attn_output = self.dropout(attn_output)
       x = self.norm1(residual + attn_output)
       ffn_output = self.ffn(x)
       return self.norm2(x + ffn_output)

class UpstreamModel(nn.Module):
   def __init__(self, input_dim=54, hidden_dim=128, embed_dim=64):
       super().__init__()
       self.reshape = lambda x: x.unsqueeze(1)
       self.encoder1 = TransformerEncoderBlock(embed_dim=input_dim, num_heads=2)
       self.encoder2 = TransformerEncoderBlock(embed_dim=input_dim, num_heads=2)
       self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim//2, bidirectional=True, batch_first=True)
       self.fc = nn.Linear(hidden_dim, embed_dim)
       self.dropout = nn.Dropout(0.5)
   def forward(self, x):
       x = self.reshape(x)
       x = self.encoder1(x)
       x = self.encoder2(x)
       x, _ = self.lstm(x)
       x = x.squeeze(1)
       x = self.fc(x)
       x = self.dropout(x)
       return x

class EmotionClassifier(nn.Module):
   def __init__(self, embed_dim=64):
       super().__init__()
       self.classifier = nn.Sequential(
           nn.Linear(embed_dim,128), nn.ReLU(), nn.Dropout(0.5),
           nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.5),
           nn.Linear(64,2)
       )
   def forward(self, x):
       return self.classifier(x)

class SpeakerClassifier(nn.Module):
   def __init__(self, embed_dim, num_speakers):
       super().__init__()
       self.grl = GradientReversalLayer()  # using GRL
       self.classifier = nn.Sequential(
           nn.Linear(embed_dim,128), nn.ReLU(), nn.Dropout(0.5),
           nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.5),
           nn.Linear(64, num_speakers)
       )
   def forward(self, x):
       x = self.grl(x)  # Gradient Reversal
       return self.classifier(x)


def evaluate_emotion(upstream, cls, loader):
   upstream.eval(); cls.eval()
   probs, labels = [], []
   with torch.no_grad():
       for feats, emo, _ in loader:
           feats = feats.to(device)
           out = cls(upstream(feats))
           prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
           probs.append(prob); labels.append(emo.numpy())
   probs = np.concatenate(probs); labels = np.concatenate(labels)
   acc = accuracy_score(labels, probs > 0.5)
   auc = roc_auc_score(labels, probs)
   return acc, auc

def evaluate_speaker_leakage(upstream, loader):
   upstream.eval()
   preds, labels = [], []
   with torch.no_grad():
       for feats, _, spk in loader:
           feats = feats.to(device)
           out = speaker_cls(upstream(feats))
           pred = out.argmax(1).cpu().numpy()
           preds.append(pred); labels.append(spk.numpy())
   preds = np.concatenate(preds); labels = np.concatenate(labels)
   return accuracy_score(labels, preds)

In [ ]:
num_speakers = df_copd_asthma_speech['sid'].nunique()
num_speakers

In [ ]:



# WandB Initialization
wandb.init(
    project="********",
    name="task1b_adv_lambda0.1",
    config={
        "model": "single_task",
        "task": "emotion_only",
        "lr": 1e-4,
        "batch_size": 32,
        "epochs": 1000,
        "seed": 42
    },
    reinit=True
)

# model initialization
upstream = UpstreamModel().to(device)
emotion_cls = EmotionClassifier().to(device)
speaker_cls = SpeakerClassifier(embed_dim=64, num_speakers=num_speakers).to(device)

# optimizer
optimizer = torch.optim.AdamW(
   list(upstream.parameters()) + list(emotion_cls.parameters()) + list(speaker_cls.parameters()),
   lr=1e-4, weight_decay=1e-4
)
criterion_emotion = nn.CrossEntropyLoss()
criterion_speaker = nn.CrossEntropyLoss()

# training loop
best_val_auc = 0.0
best_epoch = 0
best_epoch_test_auc = 0.0
best_epoch_test_jratio = 0.0

for epoch in range(1000):
   upstream.train(); emotion_cls.train(); speaker_cls.train()
   speaker_cls.grl.lambda_param = 0.1

   total_loss_emotion = 0.0
   total_loss_speaker = 0.0
   correct_emotion = 0
   correct_speaker = 0
   total_samples = 0

   for batch_features, batch_emotion, batch_speaker in train_loader:
       batch_features = batch_features.to(device)
       batch_emotion = batch_emotion.to(device)
       batch_speaker = batch_speaker.to(device)

       embeddings = upstream(batch_features)
       emotion_output = emotion_cls(embeddings)
       loss_emotion = criterion_emotion(emotion_output, batch_emotion)

       speaker_output = speaker_cls(embeddings)
       loss_speaker = criterion_speaker(speaker_output, batch_speaker)

       total_loss = loss_emotion + 1 * loss_speaker  # Adversarial Training

       optimizer.zero_grad()
       total_loss.backward()
       clip_grad_norm_(upstream.parameters(), max_norm=0.5)
       clip_grad_norm_(emotion_cls.parameters(), max_norm=0.5)
       clip_grad_norm_(speaker_cls.parameters(), max_norm=0.5)
       optimizer.step()

       total_loss_emotion += loss_emotion.item()
       total_loss_speaker += loss_speaker.item()
       correct_emotion += (emotion_output.argmax(1) == batch_emotion).sum().item()
       correct_speaker += (speaker_output.argmax(1) == batch_speaker).sum().item()
       total_samples += batch_emotion.size(0)

   #validation and testing
   val_acc, val_auc = evaluate_emotion(upstream, emotion_cls, val_loader)
   test_acc, test_auc = evaluate_emotion(upstream, emotion_cls, test_loader)
   leakage_acc = evaluate_speaker_leakage(upstream, val_loader)

   #J-ratio

   emb_val, spk_val = extract_embeddings(upstream, test_loader, device)
   j_ratio = compute_j_ratio(emb_val, spk_val)

   # Save the best model based on the validation AUC
   if val_auc > best_val_auc:
       best_val_auc = val_auc
       best_epoch_test_auc = test_auc
       best_epoch_test_jratio = j_ratio

       torch.save({
           'upstream': upstream.state_dict(),
           'emotion_cls': emotion_cls.state_dict(),
           'val_auc': val_auc,
       }, "*****_lambda0.1.pth")
       joblib.dump(scaler, "******_lambda0.1.pkl")
       best_epoch = epoch + 1


   # WandB log
   wandb.log({
       "epoch": epoch + 1,
       "train_emotion_loss": total_loss_emotion / len(train_loader),
       "train_speaker_loss": total_loss_speaker / len(train_loader),
       "train_emotion_acc": correct_emotion / total_samples,
       "train_speaker_acc": correct_speaker / total_samples,
       "val_acc": val_acc,
       "val_auc": val_auc,
       "test_acc": test_acc,
       "test_auc": test_auc,
       "speaker_leakage_acc": leakage_acc,
       "j_ratio": j_ratio,
       "best_val_auc": best_val_auc,
       "best_epoch": best_epoch,
   })

   print(f"[ADV] Epoch {epoch+1} | Val AUC: {val_auc:.4f} | Leakage: {leakage_acc:.4f} | J-ratio: {j_ratio:.4f}")


wandb.log({"final_test_auc": best_epoch_test_auc, "final_test_jratio": best_epoch_test_jratio})

wandb.finish()
print(f" Adversarial model training completed. Best Val AUC: {best_val_auc:.4f}")